> 1. Physical view: 4-node H100 deployment

```text
   ┌──────────────────────────────── LLM CLUSTER ──────────────────────────────────┐
   │                                                                               │
   ┌─────────────┐       ┌─────────────┐       ┌─────────────┐       ┌─────────────┐
   │   Node 1    │       │   Node 2    │       │   Node 3    │       │   Node 4    │
   │─────────────│       │─────────────│       │─────────────│       │─────────────│
   │  CPU + NIC  │       │  CPU + NIC  │       │  CPU + NIC  │       │  CPU + NIC  │
   │             │       │             │       │             │       │             │
   │  System     │       │  System     │       │  System     │       │  System     │
   │  RAM (DRAM) │◀──────┼─────────────┼──────▶│  RAM (DRAM) │◀──────┼─────────────┼───── ...
   │ (shared in  │ InfiniBand / RoCE   │       │ (shared in  │ InfiniBand / RoCE   │
   │   node)     │       │   node)     │       │   node)     │       │   node)     │
   │             │       │             │       │             │       │             │
   │  ┌───────┐  │       │  ┌───────┐  │       │  ┌───────┐  │       │  ┌───────┐  │
   │  │GPU0   │  │       │  │GPU0   │  │       │  │GPU0   │  │       │  │GPU0   │  │
   │  │HBM0   │  │       │  │HBM0   │  │       │  │HBM0   │  │       │  │HBM0   │  │
   │  └───────┘  │       │  └───────┘  │       │  └───────┘  │       │  └───────┘  │
   │  ┌───────┐  │       │  ┌───────┐  │       │  ┌───────┐  │       │  ┌───────┐  │
   │  │GPU1   │  │       │  │GPU1   │  │       │  │GPU1   │  │       │  │GPU1   │  │
   │  │HBM1   │  │       │  │HBM1   │  │       │  │HBM1   │  │       │  │HBM1   │  │
   │  └───────┘  │       │  └───────┘  │       │  └───────┘  │       │  └───────┘  │
   │  ┌───────┐  │       │  ┌───────┐  │       │  ┌───────┐  │       │  ┌───────┐  │
   │  │GPU2   │  │       │  │GPU2   │  │       │  │GPU2   │  │       │  │GPU2   │  │
   │  │HBM2   │  │       │  │HBM2   │  │       │  │HBM2   │  │       │  │HBM2   │  │
   │  └───────┘  │       │  └───────┘  │       │  └───────┘  │       │  └───────┘  │
   │  ┌───────┐  │       │  ┌───────┐  │       │  ┌───────┐  │       │  ┌───────┐  │
   │  │GPU3   │  │       │  │GPU3   │  │       │  │GPU3   │  │       │  │GPU3   │  │
   │  │HBM3   │  │       │  │HBM3   │  │       │  │HBM3   │  │       │  │HBM3   │  │
   │  └───────┘  │       │  └───────┘  │       │  └───────┘  │       │  └───────┘  │
   │   NVLink    │       │   NVLink    │       │   NVLink    │       │   NVLink    │
   └─────────────┘       └─────────────┘       └─────────────┘       └─────────────┘
```

Where:
* `RoCE` = [RDMA over Converged Ethernet](https://en.wikipedia.org/wiki/RDMA_over_Converged_Ethernet) - a network protocol which allows remote direct memory access (RDMA) over an Ethernet network
* `InfiniBand` = [InfiniBand](https://en.wikipedia.org/wiki/InfiniBand) - a computer networking standard used in high-performance computing that is used for data interconnect both among and within computers. InfiniBand is also used as either a direct or switched interconnect between servers and storage systems, as well as an interconnect between storage systems.
* `NIC` = Network Interface Card
* `HBM` = High Bandwidth Memory

GPU-specific memory:

* Each $GPU_i$ has its own HBM ($HBM_i$ box).

* Model weights, KV cache, and activations for that $GPU_i$ shard live there.

Shared memory per node:

* System RAM (DRAM) is shared by CPU and visible to all GPUs in the node through PCIe / NVLink.
* No memory is shared at the hardware-level; nodes communicate via network links (InfiniBand/RoCE). Memory is "shared" only logically via explicit communication (collectives, RPCs, etc.).

> 2. Logical view: how one user request uses these GPUs

How one request going to an LLM is `sharded` across a model-parallel group of GPUs (e.g., 8 GPUs across Node 1 and Node 2). The same pattern scales to more GPUs.

```text

                              ┌──────────────────── API / Frontend ───────────────────┐
User → HTTPS → Load Balancer → Inference Service → Chosen Model-Parallel Group
                              └───────────────────────────────────────────────────────┘


                     (Example model-parallel group for this request: 4 GPUs)
┌──────────────────────────────────────────── Node 1 ─────────────────────────────────────────────┐
│ CPU1 + System RAM                                                                               │
│  - Runtime for ranks on Node 1                                                                  │
│  - Token IDs (int32) for active requests                                                        │
│                                                                                                 │
│   ┌───────────────── Model Shard A ──────────────┐      ┌──────────────────────────────────────┐│
│   │ GPU_1_0                                      │      │ GPU_1_1                              ││
│   │  HBM_1_0:                                    │      │  HBM_1_1:                            ││
│   │   - Weights shard W[0]                       │      │   - Weights shard W[1]               ││
│   │   - KV cache shard KV[0]                     │      │   - KV cache shard KV[1]             ││
│   │   - Activations for this shard               │      │   - Activations for this shard       ││
│   │   - Local slice of embedded tokens           │      │   - Local slice of embedded tokens   ││
│   │     (float16/bfloat16)                       │      │     (float16/bfloat16)               ││
│   │                                              │      │                                      ││
│   └──────────────────────────────────────────────┘      └──────────────────────────────────────┘│
│                 ▲                           ▲                                        ▲          │
│                 │ NVLink                    │ NVLink                                 │          │
└─────────────────┼───────────────────────────┼────────────────────────────────────────┼──────────┘
                  │                           │                                        │  InfiniBand / RoCE
┌─────────────────┼───────────────────────────┼────────────────────────────────────────┼──────────┐
│                 ▼                           ▼                                        ▼          │
│──────────────────────────────────────────── Node 2 ─────────────────────────────────────────────│
│ CPU2 + System RAM                                                                               │
│  - Runtime for ranks on Node 2                                                                  │
│  - Token IDs (int32) for active requests                                                        │
│                                                                                                 │
│   ┌───────────────── Model Shard B ──────────────┐      ┌──────────────────────────────────────┐│
│   │ GPU_2_0                                      │      │ GPU_2_1                              ││
│   │  HBM_2_0:                                    │      │  HBM_2_1:                            ││
│   │   - Weights shard W[2]                       │      │   - Weights shard W[3]               ││
│   │   - KV cache shard KV[2]                     │      │   - KV cache shard KV[3]             ││
│   │   - Activations for this shard               │      │   - Activations for this shard       ││
│   │   - Local slice of embedded tokens           │      │   - Local slice of embedded tokens   ││
│   │     (float16/bfloat16)                       │      │     (float16/bfloat16)               ││
│   │                                              │      │                                      ││
│   └──────────────────────────────────────────────┘      └──────────────────────────────────────┘│
└─────────────────────────────────────────────────────────────────────────────────────────────────┘
```

Where:
* Local KV shard for all previous positions for this request (`K/V[i]` in HBM)
* 

Assumptions:

* Global hidden-state dimension: $d_{\text{model}}$
* Tensor-parallel group size: $G$ (here $G = 4$)
* A GPU holds $\tfrac{1}{G}$ of the hidden state and weights.
* $B$ is the size of the current batch of sequences/tokens:

    * In training: $B$ = number of training examples.
    * In inference: $B$ = number of sequences (or beams) being processed in parallel in that step;
      for a single request, $B$ can be 1.

On a $GPU_i$ in HBM there is following allocation:

* Local hidden-state shard for the current token(s):
  $$h_i \in \mathbb{R}^{B \times \frac{d_{\text{model}}}{G}}$$

* Local weight shards for the attention projections (e.g. column-parallel):
  $$W_{Q,i}, W_{K,i}, W_{V,i} \in \mathbb{R}^{\frac{d_{\text{model}}}{G} \times d_{\text{attn},i}}$$
  where $d_{\text{attn},i}$ is this GPU’s slice of the attention dimension (e.g. subset of heads).

* Local KV cache for all previous positions for this request, in this layer, on this GPU:
  $$K_i \in \mathbb{R}^{T \times d_{\text{attn},i}}, \quad
  V_i \in \mathbb{R}^{T \times d_{\text{attn},i}}$$
  where $T$ is the number of past tokens.

The request context is already encoded as the local hidden-state shard $h_i$ in HBM.

---

## 2. Local projections: GEMMs with local weight shards

On GPU $i$, the first step in the attention block is to compute $Q$, $K$, and $V$ for the current token(s) via local GEMMs:

$$Q_i = h_i W_{Q,i} \in \mathbb{R}^{B \times d_{\text{attn},i}}$$
$$K_i^{\text{new}} = h_i W_{K,i} \in \mathbb{R}^{B \times d_{\text{attn},i}}$$
$$V_i^{\text{new}} = h_i W_{V,i} \in \mathbb{R}^{B \times d_{\text{attn},i}}$$

Each is a matrix multiplication of shape
$$(B \times \tfrac{d_{\text{model}}}{G}) \cdot \left(\tfrac{d_{\text{model}}}{G} \times d_{\text{attn},i}\right)$$

Then the new keys and values are appended to the local KV cache along the time dimension:

$$K_i \leftarrow \text{concat}(K_i, K_i^{\text{new}})$$
$$V_i \leftarrow \text{concat}(V_i, V_i^{\text{new}})$$

All of this uses only the local hidden-state shard $h_i$, the local weight shards $W_{Q,i}, W_{K,i}, W_{V,i}$, and the local KV cache slice $(K_i, V_i)$.

---

## 3. Attention using the local KV cache

For the current token’s query shard $Q_i$ and GPU $i$’s cached keys $K_i$:

1. **Local attention scores** over previous positions (for this GPU’s heads):

   $$S_i = Q_i K_i^\top \in \mathbb{R}^{B \times T}$$

   This is a GEMM of shape:
   $$(B \times d_{\text{attn},i}) \cdot (d_{\text{attn},i} \times T)$$

2. **Softmax over time dimension**, usually local if each attention head is fully contained on one GPU:

   $$A_i = \text{softmax}\!\left(\frac{S_i}{\sqrt{d_k}}\right) \in \mathbb{R}^{B \times T}$$

   where $d_k$ is the per-head key dimension.

3. **Local context output** using the local value shard $V_i$:

   $$O_i = A_i V_i \in \mathbb{R}^{B \times d_{\text{attn},i}}$$

   Another GEMM:
   $$(B \times T) \cdot (T \times d_{\text{attn},i})$$

So the attention core on GPU $i$ is just:

* GEMM for scores: $Q_i K_i^\top$
* GEMM for context: $A_i V_i$

both using **only local KV cache and local weights**.

---

## 4. Output projection and residual path under tensor parallelism

The global attention output is the concatenation of $O_i$ across GPUs:

$$O = \text{concat}(O_0, O_1, \dots, O_{G-1}) \in \mathbb{R}^{B \times d_{\text{attn}}}$$

On each GPU $i$ we keep only $O_i$, and define a **row-parallel** output weight shard:

$$W_{O,i} \in \mathbb{R}^{d_{\text{attn},i} \times d_{\text{model}}}$$

Each GPU computes its partial output:

$$Z_i = O_i W_{O,i} \in \mathbb{R}^{B \times d_{\text{model}}}$$

This is again a GEMM of shape:
$$(B \times d_{\text{attn},i}) \cdot (d_{\text{attn},i} \times d_{\text{model}})$$

Then an all-reduce across the $G$ GPUs sums the partial outputs:

$$Z = \sum_{i=0}^{G-1} Z_i \in \mathbb{R}^{B \times d_{\text{model}}}$$

After the all-reduce, each GPU has the full attention output $Z$. The usual transformer operations follow:

* Residual connection:
  $$h^{\text{attn-out}} = h^{\text{in}} + Z$$
* Layer normalization, then feed-forward network (FFN), etc.

The FFN block uses the **same tensor-parallel GEMM pattern**:

* First linear: typically column-parallel across GPUs.
* Nonlinearity.
* Second linear: row-parallel plus an all-reduce over partial outputs.

---

## 5. Summary: what the single GPU “technique” is

From the standpoint of a single GPU $i$ in the model-parallel group:

1. It stores **local hidden-state shard** $h_i$, **local weight shards** $W[\cdot,i]$, and **local KV cache shard** $(K_i, V_i)$ in HBM.
2. It performs **standard GEMMs**:

   * $h_i W_{Q,i}$, $h_i W_{K,i}$, $h_i W_{V,i}$ to get local $Q_i, K_i^{\text{new}}, V_i^{\text{new}}$.
   * $Q_i K_i^\top$ and $A_i V_i$ for attention using the **local** KV cache.
   * $O_i W_{O,i}$ (and analogous FFN GEMMs) using its local weight shards.

3. It participates in a small number of **collective operations** (all-reduce / all-gather) to combine partial results into full hidden states.

So the “technique” is: **block-partitioned matrix multiplications on local shards of hidden state, weights, and KV cache**, plus collectives to reassemble global results across GPUs.


> How does GEMM works?

## 1. What is GEMM?

GEMM is the standard general matrix–matrix multiply operation:

$$
C \leftarrow \alpha A B + \beta C
$$

for matrices

* $A \in \mathbb{R}^{m \times k}$
* $B \in \mathbb{R}^{k \times n}$
* $C \in \mathbb{R}^{m \times n}$

GEMM itself is just the **operation**.
The implementation (on GPU / multi-GPU) uses **tiling** and **sharding** to break large matrices into smaller blocks that fit into fast memory and can be processed in parallel.

---

## 2. Global matrices and block structure

We can logically partition $A$, $B$, and $C$ into blocks:

$$
A =
\begin{bmatrix}
A_{00} & A_{01} \\
A_{10} & A_{11}
\end{bmatrix}
\quad
B =
\begin{bmatrix}
B_{00} & B_{01} \\
B_{10} & B_{11}
\end{bmatrix}
\quad
C =
\begin{bmatrix}
C_{00} & C_{01} \\
C_{10} & C_{11}
\end{bmatrix}
$$

ASCII view:

```text
          A (m × k)                             B (k × n)
   ┌──────────────────────┐                ┌─────────────────────┐
   │          │           │                │          │          │
   │  A_00    │   A_01    │                │   B_00   │   B_01   │
   │          │           │                │          │          │
   ├──────────┼───────────┤                ├──────────┼──────────┤
   │          │           │                │          │          │
   │  A_10    │   A_11    │                │   B_10   │   B_11   │
   │          │           │                │          │          │
   └──────────┴───────────┘                └──────────┴──────────┘

                            C = A · B  (m × n)
                         ┌─────────────────────┐
                         │          │          │
                         │   C_00   │   C_01   │
                         │          │          │
                         ├──────────┼──────────┤
                         │          │          │
                         │   C_10   │   C_11   │
                         │          │          │
                         └──────────┴──────────┘
```

---

## 3. Block matrix multiplication

For block matrices, each block $C_{ik}$ is computed as:

$$
C_{ik} = \sum_j A_{ij} B_{jk}
$$

In the $2 \times 2$ block example:

$$
\begin{aligned}
C_{00} &= A_{00} B_{00} + A_{01} B_{10}, \\
C_{01} &= A_{00} B_{01} + A_{01} B_{11}, \\
C_{10} &= A_{10} B_{00} + A_{11} B_{10}, \\
C_{11} &= A_{10} B_{01} + A_{11} B_{11}.
\end{aligned}
$$

ASCII view for $C_{00}$:

```text
      k-dimension split into tiles 0 and 1

      A row-block i=0                   B col-block k=0
   ┌──────────────────┐              ┌─────────────────┐
   │        │         │              │        │        │
   │  A_00  │  A_01   │      ×       │  B_00  │  B_10  │^T
   │        │         │              │        │        │
   └──────────────────┘              └─────────────────┘
            │                                 │
            │ GEMM: A_00·B_00                 │ GEMM: A_01·B_10
            │                                 │
            └───────────────┬─────────────────┘
                            ▼
                 C_00 = A_00·B_00 + A_01·B_10
```

Implementation sketch:

1. Load $A_{00}$ and $B_{00}$ into fast memory; compute partial $C_{00}^{(0)} = A_{00} B_{00}$.
2. Load $A_{01}$ and $B_{10}$; accumulate $C_{00} = C_{00}^{(0)} + A_{01} B_{10}$.
3. Store $C_{00}$ to global memory.

This pattern is repeated for all $C_{ik}$.

---

## 4. Within a single GPU: tiling

On a single GPU, the **full matrices** typically reside in device memory (HBM), but the GEMM kernel processes **tiles** that fit into on-chip fast memory (shared memory / registers).

Conceptually:

```text
Global A in HBM                        Tiles in SMEM/registers
┌─────────────────────┐          ┌───────────┐     ┌───────────┐
│ A_00 │ A_01 │ ...   │   =>     │  A_00     │  +  │  A_01     │  +  ...
├─────────────────────┤          └───────────┘     └───────────┘
│ A_10 │ A_11 │ ...   │
└─────────────────────┘
```

Each tile pair $(A_{\text{tile}}, B_{\text{tile}})$:

* Is loaded into fast memory,
* Processed by a thread block / warp,
* Produces a corresponding tile of $C$.

This:

* Improves locality and reuse,
* Enables high **parallelism** across many tiles.

---

## 5. Across multiple GPUs: sharding

For large models (LLMs), the **logical** weight matrix is also sharded across GPUs.
For example, column-parallel sharding:

Let the global weight be:

$$
W \in \mathbb{R}^{d_{\text{model}} \times d_{\text{out}}}
$$

We split its columns across $G$ GPUs:

$$
W =
\bigl[ W_{[0]} \quad W_{[1]} \quad  \dots \quad  W_{[G-1]} \bigr],
\quad
W_{[i]} \in \mathbb{R}^{d_{\text{model}} \times d_{\text{out},i}},
\quad
\sum_i d_{\text{out},i} = d_{\text{out}}
$$

Each GPU $i$ holds $W_{[i]}$ and computes a **local GEMM**:

$$
C_{[i]} = H , W_{[i]},
$$

where $H \in \mathbb{R}^{B \times d_{\text{model}}}$ is the (possibly sharded) hidden state.

The logical global result is then formed by concatenation (for column-parallel):

$$
C = \text{concat}\bigl(C_{[0]}, C_{[1]}, \dots, C_{[G-1]}\bigr)
\in \mathbb{R}^{B \times d_{\text{out}}}
$$

Or by summation (for row-parallel):

$$
C = \sum_{i=0}^{G-1} C_{[i]}
$$

So:

* Globally, $W$ is a single large matrix.
* Physically, it is realized as **multiple non-overlapping shards** stored on different GPUs.
* Each GPU runs GEMM on its shard in parallel.
* A collective (all-gather or all-reduce) reconstructs the logical global result.
